# Hinglish Sarcasm Detector
**NLP Mini Project** · Khushi Panchotiya (23BIT112) & Grishma Makwana (23BIT108) · PDEU

We classify code-mixed Hindi-English (Hinglish) comments as **Sarcastic** or **Sincere**.
This notebook is self-contained. It downloads the data, trains a classical baseline (TF-IDF + Logistic Regression), fine-tunes a transformer (HingBERT), and compares them on a held-out test set.

**To run on Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all (about 5–10 minutes).

## 1. Setup

In [ ]:
FAST_DEMO = False   # True = tiny quick run (1,500 training rows, 1 epoch) just to test the notebook

import re, random, numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
LABELS = ["Sincere", "Sarcastic"]
sns.set_theme(style="whitegrid")
print("GPU available:", torch.cuda.is_available())

## 2. Load the dataset
Public dataset: *Hinglish Sarcasm Detection Dataset* (Jain & Nagulapalli, 2025). We use the `human_labelled` column as the target.

In [ ]:
URL = "https://raw.githubusercontent.com/mananjen/hinglish_sarcasm_dataset/main/human_labelled_features.csv"
raw = pd.read_csv(URL, encoding="utf-8-sig")
print("Rows:", len(raw))
raw[["comment", "human_labelled"]].head(8)

## 3. Clean the data
- map label spellings to **Sincere** / **Sarcastic** and drop invalid labels
- drop texts that appear more than once with **conflicting** labels
- remove duplicates, so the same comment cannot land in both train and test
- light normalisation: collapse stretched letters (*waaaah* → *waah*), keep punctuation, emojis and stop words (they carry tone)

In [ ]:
label_map = {"sarcastic": "Sarcastic", "non-sarcastic": "Sincere", "non_sarcastic": "Sincere"}
df = pd.DataFrame({"raw": raw["comment"].fillna("").astype(str)})
df["label"] = raw["human_labelled"].astype(str).str.strip().str.lower().map(label_map)
df = df[df["label"].notna()]

df["key"] = df["raw"].str.lower().str.split().str.join(" ")
n_labels = df.groupby("key")["label"].nunique()
df = df[~df["key"].isin(n_labels[n_labels > 1].index)].drop_duplicates("key")

def normalize(t):
    t = re.sub(r"https?://\S+", " ", t)
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)      # waaaah -> waah
    return re.sub(r"\s+", " ", t).strip()

df["text"] = df["raw"].map(normalize)
df = df[["text", "label"]].reset_index(drop=True)
print("Rows after cleaning:", len(df))
print(df["label"].value_counts())

ax = sns.countplot(data=df, x="label", hue="label", order=LABELS, palette=["#3b8d6b", "#d9480f"], legend=False)
ax.set_title("Class distribution"); plt.show()

## 4. Train / validation / test split
Stratified 80 / 10 / 10. The **validation** set is used while training (to pick the best epoch); the **test** set is used only once, at the end.

In [ ]:
train, tmp = train_test_split(df, test_size=0.2, random_state=SEED, stratify=df["label"])
val, test = train_test_split(tmp, test_size=0.5, random_state=SEED, stratify=tmp["label"])
if FAST_DEMO:
    train = train.sample(1500, random_state=SEED)
print({"train": len(train), "validation": len(val), "test": len(test)})

## 5. Baseline: TF-IDF + Logistic Regression
Word and character n-grams are turned into TF-IDF features and classified with Logistic Regression. It treats words independently, so it cannot understand context, which is why we expect a transformer to do better.

In [ ]:
def show_eval(y_true, y_pred, title):
    print(f"=== {title} ===")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.3f}   Macro-F1: {f1_score(y_true, y_pred, average='macro'):.3f}\n")
    print(classification_report(y_true, y_pred, labels=LABELS, digits=3))
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    fig, ax = plt.subplots(figsize=(3.8, 3.4))
    ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(ax=ax, values_format="d", cmap="Blues", colorbar=False)
    ax.set_title(title); plt.tight_layout(); plt.show()

baseline = Pipeline([
    ("features", FeatureUnion([
        ("word", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                                 token_pattern=r"(?u)\b\w+\b")),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
    ])),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", C=2.0, random_state=SEED)),
])
baseline.fit(train["text"], train["label"])
tfidf_pred = baseline.predict(test["text"])
show_eval(test["label"], tfidf_pred, "TF-IDF + Logistic Regression (test)")

## 6. Fine-tune HingBERT
**HingBERT** (`l3cube-pune/hing-bert`) is a BERT model pretrained on Hinglish text. We add a 2-class classification layer and fine-tune the whole network.

| Setting | Value |
|---|---|
| Epochs | up to 5, early stopping after 2 epochs without improvement |
| Learning rate | 2e-5 |
| Batch size | 16 |
| Max length | 128 tokens |
| Selection | epoch with the best **validation** macro-F1 |

In [ ]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding,
                          Trainer, TrainingArguments, EarlyStoppingCallback, set_seed)
set_seed(SEED)
MODEL_NAME = "l3cube-pune/hing-bert"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
label2id = {l: i for i, l in enumerate(LABELS)}; id2label = {i: l for l, i in label2id.items()}

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, frame):
        self.enc = tok(frame["text"].tolist(), truncation=True, max_length=128)
        self.labels = frame["label"].map(label2id).tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item

train_ds, val_ds, test_ds = TextDataset(train), TextDataset(val), TextDataset(test)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, label2id=label2id, id2label=id2label)

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "macro_f1": f1_score(p.label_ids, preds, average="macro")}

EPOCHS = 1 if FAST_DEMO else 5
steps = -(-len(train_ds) // 16) * EPOCHS
args = TrainingArguments(
    output_dir="hingbert_out", num_train_epochs=EPOCHS, learning_rate=2e-5,
    per_device_train_batch_size=16, per_device_eval_batch_size=64, weight_decay=0.01,
    warmup_steps=int(0.1 * steps), eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1", save_total_limit=1,
    seed=SEED, fp16=torch.cuda.is_available(), report_to="none", logging_strategy="epoch")
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer.train()

### Training curve (validation macro-F1 per epoch)

In [ ]:
log = pd.DataFrame(trainer.state.log_history)
val_log = log[log["eval_macro_f1"].notna()][["epoch", "eval_loss", "eval_accuracy", "eval_macro_f1"]]
display(val_log.round(3).set_index("epoch"))
ax = val_log.plot(x="epoch", y=["eval_macro_f1", "eval_accuracy"], marker="o", figsize=(5, 3))
ax.set_title("Validation metrics"); ax.set_ylabel("score"); plt.show()

## 7. Test the fine-tuned model

In [ ]:
logits = trainer.predict(test_ds).predictions
hb_pred = [id2label[i] for i in logits.argmax(-1)]
show_eval(test["label"], hb_pred, "HingBERT (test)")

## 8. Comparison

In [ ]:
y = test["label"].values
majority = np.array([train["label"].mode()[0]] * len(y))
results = pd.DataFrame({
    "Majority class": [accuracy_score(y, majority), f1_score(y, majority, average="macro")],
    "TF-IDF + LR":    [accuracy_score(y, tfidf_pred), f1_score(y, tfidf_pred, average="macro")],
    "HingBERT":       [accuracy_score(y, hb_pred), f1_score(y, hb_pred, average="macro")],
}, index=["Accuracy", "Macro-F1"]).T.round(3)
display(results)

ax = results["Macro-F1"].sort_values().plot.barh(color=["#9aa4b2", "#9aa4b2", "#3b5bdb"], figsize=(6, 2.8))
for p in ax.patches:
    ax.text(p.get_width() + 0.01, p.get_y() + p.get_height() / 2, f"{p.get_width():.3f}", va="center")
ax.set_xlim(0, 1); ax.set_title("Macro-F1 on the test set"); plt.show()

## 9. Try it on your own sentences

In [ ]:
sentences = [
    "wah kya baat hai, bilkul time pe aaye",
    "thanks yaar, time pe aaye, maza aa gaya",
    "great, another Monday meeting, just what I wanted",
    "Kya shandar service hai, 2 ghante se wait kar raha hu",
]
model.eval()
enc = tok([normalize(s) for s in sentences], return_tensors="pt", padding=True, truncation=True, max_length=128).to(model.device)
with torch.no_grad():
    probs = torch.softmax(model(**enc).logits, -1).cpu().numpy()
pd.DataFrame({"comment": sentences,
              "HingBERT": [id2label[i] for i in probs.argmax(-1)],
              "confidence": probs.max(-1).round(2),
              "TF-IDF + LR": baseline.predict([normalize(s) for s in sentences])})

## 10. Summary
- Data: public Hinglish sarcasm dataset, cleaned and split 80/10/10 (stratified, seed 42).
- Baseline: TF-IDF + Logistic Regression. Model: fine-tuned HingBERT.
- Metric: **macro-F1** (the classes are slightly imbalanced), plus accuracy, per-class precision/recall and confusion matrices.
- In our full project we also fine-tuned MuRIL and XLM-RoBERTa and compared all models. The best model, HingBERT, reached about 0.69 macro-F1 on the test set, below our 0.80 target. Reasons: noisy single-annotator labels and comments with no context.
- Limitation: sarcasm is subjective and depends on context, so a single comment is often ambiguous.